In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_DIR = Path("..") / "data" / "pass_decision_h256_n0.05_seed0_id2607210153"
H5_NAME = "gaussian.h5"  # continuous activity; use data.h5 for poisson counts
AREAS = ["P", "D"]

h5_path = RUN_DIR / H5_NAME
if not h5_path.is_file():
    raise FileNotFoundError(h5_path)

activity = {}
with h5py.File(h5_path, "r") as f:
    sess = f["0"]
    for area in AREAS:
        key = f"area-{area}"
        activity[area] = np.asarray(sess[key], dtype=np.float64)  # (trials, time, neurons)

print(f"loaded {h5_path}")
for area, x in activity.items():
    print(f"  area-{area}: shape={x.shape}")


In [ ]:
# Global mean / high (max) / low (min) over all trials × time × neurons
rows = []
for area, x in activity.items():
    rows.append(
        {
            "area": area,
            "mean": float(np.mean(x)),
            "low": float(np.min(x)),
            "high": float(np.max(x)),
            "std": float(np.std(x)),
            "n_trials": int(x.shape[0]),
            "n_time": int(x.shape[1]),
            "n_neurons": int(x.shape[2]),
        }
    )

summary = pd.DataFrame(rows).set_index("area")
display(summary)


In [ ]:
# Per-timestep: trial-averaged population mean, plus high/low across neurons
fig, axes = plt.subplots(1, len(AREAS), figsize=(12, 4), dpi=150, sharey=True)
if len(AREAS) == 1:
    axes = [axes]

for ax, area in zip(axes, AREAS):
    x = activity[area]  # (trials, time, neurons)
    # average over trials first → (time, neurons)
    trial_mean = x.mean(axis=0)
    pop_mean = trial_mean.mean(axis=1)  # (time,)
    high = trial_mean.max(axis=1)
    low = trial_mean.min(axis=1)
    t = np.arange(pop_mean.shape[0])

    ax.fill_between(t, low, high, color="C0", alpha=0.2, label="low–high (neurons)")
    ax.plot(t, pop_mean, color="C0", lw=1.5, label="mean")
    ax.set_title(f"area-{area}")
    ax.set_xlabel("time")
    ax.grid(axis="y", color="0.9", lw=0.8)
    ax.set_axisbelow(True)

axes[0].set_ylabel("activity")
axes[0].legend(loc="upper right", fontsize=9)
fig.suptitle(f"{RUN_DIR.name} — {H5_NAME}", fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# Per-neuron mean activity (avg over trials × time), with area-level high/low marked
fig, axes = plt.subplots(1, len(AREAS), figsize=(12, 4), dpi=150, sharey=False)
if len(AREAS) == 1:
    axes = [axes]

for ax, area in zip(axes, AREAS):
    x = activity[area]
    neuron_mean = x.mean(axis=(0, 1))  # (neurons,)
    order = np.argsort(neuron_mean)
    y = neuron_mean[order]

    ax.plot(y, color="0.35", lw=1.0)
    ax.axhline(y.mean(), color="C0", ls="-", lw=1.25, label=f"mean={y.mean():.4f}")
    ax.axhline(y.min(), color="C1", ls="--", lw=1.0, label=f"low={y.min():.4f}")
    ax.axhline(y.max(), color="C3", ls="--", lw=1.0, label=f"high={y.max():.4f}")
    ax.set_title(f"area-{area} neuron means")
    ax.set_xlabel("neuron (sorted)")
    ax.legend(fontsize=8, loc="best")
    ax.grid(axis="y", color="0.9", lw=0.8)
    ax.set_axisbelow(True)

axes[0].set_ylabel("mean activity")
fig.suptitle(f"{RUN_DIR.name} — per-neuron mean / high / low", fontsize=13)
fig.tight_layout()
plt.show()


## Poisson sampling: unscaled vs zscore scaled

Continuous activity is near-zero mean; D high/low ≈ ±1.25 exceeds the assumed `[-1, 1]` rate map.
`sample_poisson_counts` can zscore scale activity into `[-1, 1]` before `rate = rate_max * (x+1)/2`.

CSV from:
```bash
python pd_compute_decodability.py data --pattern pass_decision_h256_n0.05_seed0_id2607210153 \
  --output data/pass_decision_h256_n0.05_seed0_id2607210153/pd_decodability_scale_compare.csv
```




In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CSV = (
    Path("..")
    / "data"
    / "pass_decision_h256_n0.05_seed0_id2607210153"
    / "pd_decodability_scale_compare.csv"
)
df = pd.read_csv(CSV)

POISSON_PAIRS = [
    ("score_poisson_dt0.01_rate20", "score_poisson_scaled_dt0.01_rate20", "dt=0.01 rate=20"),
    ("score_poisson_dt0.01_rate40", "score_poisson_scaled_dt0.01_rate40", "dt=0.01 rate=40"),
    ("score_poisson_dt0.05_rate40", "score_poisson_scaled_dt0.05_rate40", "dt=0.05 rate=40"),
]
TARGETS = ["inp", "cumsum", "sign_cumsum"]
AREAS = ["P", "D"]

fig, axes = plt.subplots(len(AREAS), len(POISSON_PAIRS), figsize=(14, 7), dpi=150, sharey=True)
x = np.arange(len(TARGETS))
width = 0.35

for row, area in enumerate(AREAS):
    sub = df[df["decode_from"] == area].set_index("target")
    for col, (unscaled_c, scaled_c, title) in enumerate(POISSON_PAIRS):
        ax = axes[row, col]
        u = [float(sub.loc[t, unscaled_c]) for t in TARGETS]
        s = [float(sub.loc[t, scaled_c]) for t in TARGETS]
        ax.bar(x - width / 2, u, width, label="unscaled", color="C0", edgecolor="black", lw=0.5)
        ax.bar(x + width / 2, s, width, label="zscore scaled", color="C1", edgecolor="black", lw=0.5)
        cont = [float(sub.loc[t, "score_continuous"]) for t in TARGETS]
        ax.scatter(x, cont, color="black", zorder=3, s=28, label="continuous")
        ax.axhline(0, color="0.5", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(TARGETS, fontsize=10)
        ax.set_axisbelow(True)
        ax.grid(axis="y", color="0.9", lw=0.8)
        if row == 0:
            ax.set_title(title, fontsize=12)
        if col == 0:
            ax.set_ylabel(f"area-{area}\n$R^2$", fontsize=12)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=10, frameon=False)
fig.suptitle("Poisson decodability: unscaled vs zscore scaled", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

rows = []
for _, r in df.iterrows():
    for unscaled_c, scaled_c, title in POISSON_PAIRS:
        rows.append(
            {
                "area": r["decode_from"],
                "target": r["target"],
                "config": title,
                "unscaled": r[unscaled_c],
                "scaled": r[scaled_c],
                "delta (scaled-unscaled)": r[scaled_c] - r[unscaled_c],
            }
        )
delta = pd.DataFrame(rows)
display(delta.pivot_table(index=["area", "target"], columns="config", values="delta (scaled-unscaled)"))


